# Regularization

模型太复杂会记住噪声（过拟合）。正则化 = 给"复杂度"付代价。本课用多项式回归演示 L1/L2，再实现 Dropout 与早停。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 过拟合演示：高次多项式


真实函数 $y = \sin x$，数据含噪声。次数 15 的多项式能穿过所有训练点，但在点之间剧烈振荡——典型的过拟合。


In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(-3, 3, 40)
y = np.sin(x) + 0.15*rng.standard_normal(40)
xs = np.linspace(-3, 3, 300)

# 封闭解岭回归：(XᵀX + λI)⁻¹Xᵀy
def ridge_poly(x, y, deg, lam, grid):
    V = np.vander(x, deg+1, increasing=True)
    A = V.T @ V + lam*np.eye(deg+1)
    w = np.linalg.solve(A, V.T @ y)
    return np.vander(grid, deg+1, increasing=True) @ w

plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=10, alpha=0.6, label='数据')
plt.plot(xs, np.sin(xs), 'k-', lw=2, label='真实 sin(x)')
plt.plot(xs, ridge_poly(x, y, 15, 0.0, xs), 'r--', lw=1.5, label='次数15 无正则（过拟合）')
plt.plot(xs, ridge_poly(x, y, 15, 1e-2, xs), 'b-', lw=1.5, label='次数15 + L2 λ=0.01')
plt.xlabel('x'); plt.ylabel('y')
plt.title('过拟合 vs L2 正则')
plt.legend(); plt.grid(alpha=0.3)


## 2. L1 vs L2：稀疏 vs 收缩


- **L2（权重衰减）**：$\lambda\|w\|_2^2$，把大权重按比例**收缩**，不置零
- **L1**：$\lambda\|w\|_1$，鼓励**稀疏**（很多权重恰好为 0）——特征选择


In [ ]:
V = np.vander(x, 16, increasing=True)[:, 1:]    # 去掉常数列（全 1，std=0 会除零）
Vn = V / V.std(axis=0)                          # 标准化特征（数值稳定）
Vt = torch.tensor(Vn, dtype=torch.float64); yt = torch.tensor(y, dtype=torch.float64)

# L2 封闭解
d = Vn.shape[1]
w_l2 = torch.linalg.solve(Vt.T@Vt + 0.01*torch.eye(d, dtype=torch.float64), Vt.T@yt)

# L1：梯度下降（近端/次梯度简化版）
w_l1 = torch.zeros(d, dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([w_l1], lr=1e-3)
for _ in range(4000):
    opt.zero_grad()
    loss = ((Vt@w_l1 - yt)**2).mean() + 0.05*w_l1.abs().sum()
    loss.backward(); opt.step()

print(f"L1 非零系数数: {(w_l1.abs().detach().numpy() > 1e-3).sum()} / {d}")
print(f"L2 非零系数数: {(w_l2.abs().numpy() > 1e-3).sum()} / {d}")
print("→ L1 把多数系数压到 0，L2 只收缩不置零")


## 3. Dropout：训练时随机失活


训练时以概率 $p$ 把神经元置 0（并除以 $1-p$ 保持期望不变），等价于训练"子网络的集成"；推理时全部保留。

$$\mathbb{E}[\text{Dropout}(x)] = x \quad(\text{因为 } x\cdot(1-p)/(1-p))$$


In [ ]:
# 手写 Dropout 验证期望
rng = np.random.default_rng(1)
x = np.array([1.0, 2.0, 3.0, 4.0])
p = 0.5
sums = []
for _ in range(100000):
    mask = (rng.random(4) > p).astype(float) / (1-p)     # 失活后缩放
    sums.append((x*mask).mean())
print(f"E[Dropout(x)] 平均 = {np.mean(sums):.4f}，x 均值 = {x.mean():.4f}")
print("→ 缩放 1/(1-p) 保证期望不变")


In [ ]:
# Dropout 对泛化的影响：同一模型带/不带 Dropout
def make_batches(X, y, bs, shuffle=True, seed=0):
    r = np.random.default_rng(seed)
    idx = np.arange(len(X))
    if shuffle:
        r.shuffle(idx)
    for i in range(0, len(idx), bs):
        sel = idx[i:i+bs]
        yield X[sel], y[sel]

def train(model, opt, Xtr, ytr, Xva, yva, epochs=100, bs=32, seed=0):
    tr_loss, va_loss = [], []
    xtr = torch.tensor(Xtr, dtype=torch.float32); ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    xva = torch.tensor(Xva, dtype=torch.float32); yva_t = torch.tensor(yva, dtype=torch.float32).unsqueeze(1)
    for epoch in range(epochs):
        model.train()
        ep = []
        for xb, yb in make_batches(xtr, ytr_t, bs, seed=seed+epoch):
            opt.zero_grad()
            loss = F.binary_cross_entropy(torch.sigmoid(model(xb)), yb)
            loss.backward(); opt.step()
            ep.append(loss.item())
        model.eval()
        with torch.no_grad():
            vl = F.binary_cross_entropy(torch.sigmoid(model(xva)), yva_t).item()
        tr_loss.append(np.mean(ep)); va_loss.append(vl)
    return tr_loss, va_loss

def run(dropout_p, seed=0):
    torch.manual_seed(0)
    layers = [nn.Linear(2, 32), nn.ReLU()]
    if dropout_p:
        layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(32, 1)]
    model = nn.Sequential(*layers)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    _, va_curve = train(model, opt, X[tr], y[tr], X[va], y[va], epochs=150)
    return va_curve

# 复用上一课的数据划分（若未定义则重新生成）
try:
    X, y, tr, va, te
except NameError:
    rng = np.random.default_rng(42)
    n = 400
    X0 = rng.standard_normal((n, 2)) + np.array([-2.0, 0.0])
    X1 = rng.standard_normal((n, 2)) + np.array([2.0, 0.0])
    X = np.vstack([X0, X1]); y = np.concatenate([np.zeros(n), np.ones(n)])
    idx = rng.permutation(len(X))
    n_tr = int(0.7*len(X)); n_va = int(0.15*len(X))
    tr, va, te = idx[:n_tr], idx[n_tr:n_tr+n_va], idx[n_tr+n_va:]

va0 = run(0.0)
va1 = run(0.3)

plt.figure(figsize=(8, 4.5))
plt.plot(va0, label='val 无 Dropout')
plt.plot(va1, label='val Dropout=0.3')
plt.xlabel('epoch'); plt.ylabel('val loss')
plt.title('Dropout 降低验证损失（抑制过拟合）')
plt.legend(); plt.grid(alpha=0.3)


## 4. 早停与数据增强


- **早停（Early Stopping）**：val 损失不再下降就停止，保存最优权重——"免费的正则化"
- **数据增强**：对输入做保标签变换（图像翻转/旋转/加噪），扩大有效样本量
- 两者都是在"减少过拟合"和"不损失拟合能力"之间找平衡


In [ ]:
# 早停演示：监控 val，连续 15 轮不降就停
def train_early_stop(X, y, tr, va, patience=15, seed=0):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    xtr = torch.tensor(X[tr], dtype=torch.float32); ytr = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1)
    xva = torch.tensor(X[va], dtype=torch.float32); yva = torch.tensor(y[va], dtype=torch.float32).unsqueeze(1)
    best, wait, best_state = float('inf'), 0, None
    xtr = torch.tensor(X[tr], dtype=torch.float32); ytr_t = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1)
    for epoch in range(300):
        model.train()
        for xb, yb in make_batches(xtr, ytr_t, 32, seed=seed+epoch):
            opt.zero_grad()
            loss = F.binary_cross_entropy(torch.sigmoid(model(xb)), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = F.binary_cross_entropy(torch.sigmoid(model(xva)), yva).item()
        if vl < best - 1e-4:
            best, wait = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                model.load_state_dict(best_state)
                return epoch, best
    return epoch, best

stop_ep, best_vl = train_early_stop(X, y, tr, va)
print(f"早停在 epoch {stop_ep} 停止，最优 val loss = {best_vl:.4f}")
print("→ 早停 = 训练过程中自动选『恰好不过拟合』的时刻")


## 5. 权重衰减与 AdamW


L2 正则在 SGD 里等于"权重衰减"：$w \leftarrow (1-\eta\lambda)w - \eta\nabla L$。但 Adam 的自适应学习率会破坏这个等价——**AdamW** 把权重衰减放在动量之外，恢复 L2 语义。工程上：用 AdamW 代替 Adam+L2。


## 课后练习


1. **λ 扫描**：λ ∈ [0, 1e-4, 1e-2, 1] 重跑岭回归，观察曲线从过拟合到欠拟合的转变。
2. **L1 稀疏性**：把 L1 惩罚从 0.05 调到 0.5，统计非零系数变化。
3. **Dropout 位置**：比较"隐藏层前"与"输入层后"加 Dropout 的效果差异。
4. **早停敏感性**：把 patience 改成 5 与 50，观察停止点与最终性能。
5. **思考**：为什么说早停是"免费"的正则化？它等价于限制了什么？
